# M21 — Train a Neural Network as a Black Box

**Objective:** train a neural network end-to-end before opening its internals.

M20 established a monitored optimizer policy: declared seeds, learning-rate
awareness, loss/update monitoring, and rollback when a named contract fails.
M21 applies that discipline to a **complete neural estimator**. The useful whole
is not a neuron equation. It is:

`dataset → stratified hold-out → training-only StandardScaler → declared MLP → fit → loss_curve / validation_scores → held-out accuracy, macro F1, confusion_matrix`

Weight matrices, activations, a manual forward pass, gradients, and
backpropagation stay closed. Those are M22–M24.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log. A
prediction is falsifiable: name a metric or curve direction and a bound, not a
vibe.

Do not inspect `coefs_`, `intercepts_`, activations, a reconstructed forward
pass, or gradients. If a failure can be diagnosed from split, labels, budget,
seed, loss, validation, or the confusion matrix, stay at that level.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M21" / "training_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M21.training_core import (
    DEFAULT_HIDDEN_UNITS,
    DEFAULT_LEARNING_RATE,
    DEFAULT_MAX_ITER,
    DEFAULT_MODEL_SEED,
    DEFAULT_SPLIT_SEED,
    DEFAULT_TEST_SIZE,
    compact_report,
    dataset_summary,
    inspect_holdout_split,
    most_confused_pair,
    print_run_evidence,
    train_black_box,
)

print("repository root:", ROOT)
print("frozen split_seed:", DEFAULT_SPLIT_SEED)
print("frozen model_seed:", DEFAULT_MODEL_SEED)
print("frozen test_fraction:", DEFAULT_TEST_SIZE)
print("frozen hidden_units:", DEFAULT_HIDDEN_UNITS)
print("frozen max_iter:", DEFAULT_MAX_ITER)
print("frozen learning_rate_init:", DEFAULT_LEARNING_RATE)
print("preprocessing: StandardScaler on training data only through a Pipeline")


## M20 boundary: keep the optimizer discipline, close the neuron

M20 already forced you to name seeds, budgets, and rollback triggers. M21
reuses that habit on a real classifier.

What this mission **opens:** whole-network training configuration, the
stratified split, training budget, model seed, capacity, and training-label
integrity.

What stays **deferred:**
- M22 — neuron, bias, activation functions, dense-layer shapes
- M23 — NumPy forward-pass reconstruction
- M24 — backpropagation / credit assignment
- M25 — a hand-authored PyTorch loop

The `relu` string inside the harness is a frozen scikit-learn knob, not a
lesson. Do not descend into it here.


## Frozen black-box configuration

Declare the useful whole **before** the first fit:

| Control | Reference value |
| --- | --- |
| Dataset | bundled `sklearn.datasets.load_digits` (offline, 8×8 pixels flattened to 64 features, 10 digit classes) |
| Split | `split_seed=2101`, `test_fraction=0.25`, stratified |
| Preprocessing | `StandardScaler` fitted on **training data only** inside a `Pipeline` |
| Estimator | `MLPClassifier`, `hidden_units=64`, solver `adam`, `learning_rate_init=0.001`, `batch_size=64` |
| Budget | `max_iter=60` |
| Early stopping | `validation_fraction=0.15` taken from the **training** split; test data stays out |
| Model seed | `2101` |

Primary source for the estimator/pipeline API is the official scikit-learn
user guide (`sklearn-guide` in `data/source_registry.json`). Do not open
3b1b, micrograd, or PyTorch on this mission.


## Predict before running — dataset contract

Timestamp a prediction before `inspect-dataset`.

The digits fixture is bundled with scikit-learn. Predict:
- `samples` is about 1800, `features` is 64, `classes` is 10
- class counts are roughly balanced, so a majority baseline will sit near 0.10
- pixel values live on a small non-negative scale, so scaling is optional for
  identifiability but is part of the declared pipeline

If your prediction is “it is MNIST downloaded from the internet,” write that
down so the observation can falsify it.


In [ ]:
summary = dataset_summary()
print("dataset_summary", summary)
assert summary["samples"] == 1797
assert summary["features"] == 64
assert summary["classes"] == 10


### Observe the declared data contract

The fixture is local. There is no download, no secret, and no paid API. Ten
roughly balanced digit classes make a majority-class classifier a weak but
honest baseline. That baseline is the first number a training run must beat.


## Hold-out contract

The test split is **not** a second training set. It does not fit the scaler,
does not fit the network, and does not participate in early stopping.

sklearn's `validation_fraction` slices the **training** rows only. If a later
claim treats the 450-row test set as if it chose `n_iter_`, the claim is
invalid even if the headline accuracy looks good.


## Predict before running — majority baseline on the held-out split

Timestamp a prediction before `inspect-holdout`.

Predict:
- `train_size` ≈ 0.75 × 1797 and `test_size` ≈ 0.25 × 1797
- `majority_baseline_accuracy` < 0.11 because the ten test classes stay
  roughly balanced under stratification
- no network has been fitted yet, so this number is a data-split property,
  not a model property


In [ ]:
holdout = inspect_holdout_split()
print("holdout_split", holdout)
print("majority_baseline_accuracy", holdout["majority_baseline_accuracy"])
print("StandardScaler policy:", holdout["preprocessing"])
assert holdout["train_size"] + holdout["test_size"] == 1797
assert holdout["majority_baseline_accuracy"] < 0.11


### Observe before the first fit

You now have a measured majority baseline from the **same** split the
reference run will use. Keep that number visible. Every later accuracy claim
is “better than this baseline on this 450-row test set,” not “the network
understood digits.”


## Predict before running — reference black-box run

Timestamp a prediction before `train-reference`.

Frozen config: `split_seed=2101`, `StandardScaler` on training data only,
`hidden_units=64`, `max_iter=60`, `learning_rate_init=0.001`,
`model_seed=2101`, early stopping on a training-internal 15% slice.

Predict:
- `test_accuracy` and `macro_f1` both exceed 0.90
- `loss_curve` has more than one point and the last loss is below the first
- `validation_scores` is non-empty because early stopping is on
- `test_accuracy` beats `majority_baseline_accuracy` by a wide margin

Do not predict a neuron-level story.


In [ ]:
reference = train_black_box()
print_run_evidence(reference, label="reference")
print("beats majority baseline:", reference.test_accuracy > reference.majority_baseline_accuracy)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(reference.loss_curve, marker="o")
axes[0].set(xlabel="training iteration", ylabel="training loss", title="Reference loss_curve")
axes[1].plot(reference.validation_scores, marker="o")
axes[1].set(xlabel="training iteration", ylabel="validation score", title="Reference validation_scores")
fig.tight_layout()
plt.show()
print("n_loss_curve_points", len(reference.loss_curve))
print("n_validation_scores", len(reference.validation_scores))
print("first_loss", reference.loss_curve[0], "final_loss", reference.final_loss)


### Interpret only observable metrics

Supported claims:
- the run beat the measured majority baseline on held-out data
- training loss changed across iterations
- a training-internal validation score existed and selected an iterate

Unsupported claims:
- a particular hidden unit detected loops
- ReLU “caused” the accuracy
- the test set chose the stopping iterate

`train_accuracy` is scored on the full training matrix, including the 15%
slice sklearn used for early stopping. Treat it as fit-side evidence, not as
a second hold-out.


## Predict before running — held-out confusion matrix

Timestamp a prediction before `inspect-error-profile`.

Predict:
- the confusion_matrix is 10×10 and its entries sum to `test_size`
- the largest off-diagonal pair is a plausible digit confusion (for example
  3/8 or 1/7), not a collapse onto one class
- the most-confused pair names **true class, predicted class, count** without
  referring to a weight


In [ ]:
true_class, predicted_class, count = most_confused_pair(reference)
print("confusion_matrix sum", sum(map(sum, reference.confusion_matrix)))
print("most_confused_pair", (true_class, predicted_class, count))

fig, ax = plt.subplots(figsize=(5.2, 4.6))
image = ax.imshow(reference.confusion_matrix, interpolation="nearest")
ax.set(
    xlabel="predicted class",
    ylabel="true class",
    title="Held-out confusion_matrix",
    xticks=range(len(reference.classes)),
    yticks=range(len(reference.classes)),
)
fig.colorbar(image, ax=ax, fraction=0.046)
fig.tight_layout()
plt.show()


### Error structure is still a black-box observation

A confusion matrix localizes which **classes** the system mixes up. That is
enough to decide whether the run is acceptable evidence. It is not a license
to inspect a hidden layer. M22 can open the layer only after this error
profile is defended.


## Predict before running — same-seed replay

Timestamp a prediction before `run-replay`.

Predict that `train_black_box()` with every default unchanged equals the
`reference` object **exactly**: same metrics, same `loss_curve`, same
`validation_scores`, same `confusion_matrix`. If replay fails, the seed
contract is broken and later comparisons are not controlled.


In [ ]:
replay = train_black_box()
assert replay == reference
print("same-seed replay is exact")
print("replay compact_report", compact_report(replay))


### Replay supports the declared seed contract

Exact equality is stronger than “accuracy looked similar.” Keep the replay
even when the headline metric is already high. A different machine/BLAS
stack can theoretically perturb floats; on this CPU fixture the contract is
exact equality.


## Predict before running — different model seed

Timestamp a prediction before `run-seed-change`.

Change **only** `model_seed` from 2101 to 2102. Predict:
- the `TrainingRun` is not equal to `reference`
- `test_accuracy` still exceeds 0.90
- you will **not** keep only the luckier seed; both traces stay in the log


In [ ]:
seed_change = train_black_box(model_seed=2102)
print_run_evidence(seed_change, label="model_seed=2102")
assert seed_change != reference
print("seed-change still useful:", seed_change.test_accuracy > 0.90)
print("accuracy delta vs reference", round(seed_change.test_accuracy - reference.test_accuracy, 4))


### Interpret without cherry-picking

A second seed is sensitivity evidence, not a search for a nicer headline.
Report both runs. Do not invent an internal explanation for why 2102 differs
from 2101.


## Predict before running — Controlled failure A: one-iteration undertraining

Timestamp a prediction before `run-undertraining`.

Change **only** `max_iter` from 60 to 1. Predict:
- `test_accuracy` falls below 0.60
- the gap `reference.test_accuracy - under.test_accuracy` exceeds 0.40
- `loss_curve` is a single point or a very short trace
- split, scaler, hidden units, learning rate, and seed stay at the reference
  values


In [ ]:
under = train_black_box(max_iter=1)
print_run_evidence(under, label="max_iter=1")
print("accuracy gap vs reference", round(reference.test_accuracy - under.test_accuracy, 4))


### Diagnose before repair

The isolated cause is training budget. The gradient story is not available
yet and is not required. A repair that changes hidden units, labels, or the
split is rejected even if accuracy recovers.

Record why the one-point (or short) `loss_curve` plus the known `max_iter=1`
change is enough to name the failure.


## Predict before running — smallest repair for undertraining

Timestamp a prediction before `run-undertraining-repair`.

Predict that restoring **only** the reference `max_iter` (the default)
replays the original `reference` object exactly. No other knob moves.


In [ ]:
under_repair = train_black_box()
assert under_repair == reference
print("undertraining repair restored the reference run")


### Repair is restoration, not redesign

The safe configuration is the frozen reference. Budget failures roll back to
that configuration. They do not become an excuse to open M22 internals.


## Predict before running — tiny-capacity comparison

Timestamp a prediction before `run-capacity`.

Change **only** `hidden_units` from 64 to 4. Predict:
- `test_accuracy` is at least 0.10 worse than the reference
- the run may still beat majority baseline
- you will **not** conclude that larger networks are universally better


In [ ]:
tiny = train_black_box(hidden_units=4)
print_run_evidence(tiny, label="hidden_units=4")
print("capacity gap vs reference", round(reference.test_accuracy - tiny.test_accuracy, 4))


### Capacity is a trade-off on this fixture

This comparison is one dataset, one split, one budget, one optimizer. It
justifies only a fixture-bounded claim: four hidden units were weaker than
64 here. It does not justify “always add units.”


## Predict before running — Controlled failure B: shuffled training labels

Timestamp a prediction before `run-label-corruption`.

Permute **only** the training labels with `label_seed=2121`. Test labels stay
correct. Predict:
- `test_accuracy` falls near `majority_baseline_accuracy` (within about 0.06)
- a longer budget or a bigger network would not be an acceptable repair
- the confusion_matrix will not show a sharp diagonal


In [ ]:
bad_labels = train_black_box(shuffle_labels=True, label_seed=2121)
print_run_evidence(bad_labels, label="shuffled training labels")
print(
    "distance from majority baseline",
    round(abs(bad_labels.test_accuracy - bad_labels.majority_baseline_accuracy), 4),
)


### Diagnose target integrity failure

The features, split, scaler, architecture knobs, optimizer, and model seed
are unchanged. The mapping from training example to training label is
broken. Near-baseline held-out accuracy is the expected symptom.

Do not open the network. Do not “fix” this by adding hidden units.


## Predict before running — smallest repair for corrupted labels

Timestamp a prediction before `run-label-repair`.

Predict that restoring the original training labels (defaults, no shuffle)
replays `reference` exactly.


In [ ]:
label_repair = train_black_box()
assert label_repair == reference
print("label-integrity repair restored the reference run")


### Data-system failures stay at the data-system layer

Rollback means restore the target contract, then rerun the frozen
configuration. Architecture stories are out of scope until labels, split,
and budget are honest.


## Compare declared runs

The comparison table is whole-system evidence: baseline, budget, capacity,
seed, label integrity, traces, and held-out metrics. It is not a ranking of
neuron designs.


In [ ]:
runs = {
    "reference": reference,
    "seed_2102": seed_change,
    "max_iter_1": under,
    "hidden_units_4": tiny,
    "shuffled_labels": bad_labels,
}
print("majority_baseline_accuracy (holdout)", holdout["majority_baseline_accuracy"])
for name, run in runs.items():
    report = compact_report(run)
    print(name, report)
    print(
        " ",
        "loss_curve_len",
        report["n_loss_curve_points"],
        "validation_scores_len",
        report["n_validation_scores"],
        "confusion_matrix_sum",
        sum(map(sum, run.confusion_matrix)),
    )


## Code reading — split → scale → fit → held-out evaluation

Read `train_black_box` in `missions/M21/training_core.py` (see also
`missions/M21/code_reading.md`). Before running the next cell, mark in your
notes:

1. which call creates the stratified split
2. where `StandardScaler` is allowed to see data
3. which object is the `fit` call
4. where `X_test` first appears
5. which evidence is assembled **after** predict-on-test

Do not inspect weight attributes. Predict what happens if someone moved
`validation_fraction` onto the test rows — then confirm the source does not
do that.


In [ ]:
source = inspect.getsource(train_black_box)
markers = (
    "train_test_split",
    "StandardScaler",
    "Pipeline",
    ".fit(",
    "X_test",
    "confusion_matrix",
    "validation_fraction",
)
print("train_black_box orchestration markers")
for marker in markers:
    print(f"  {marker!r} present: {marker in source}")

print("holdout train/test sizes:", holdout["train_size"], holdout["test_size"])
print("reference train/test sizes:", reference.train_size, reference.test_size)
print("early stopping uses training split only:", holdout["early_stopping_split"])


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes for every governed action
- the measured majority baseline from the hold-out **before** the first fit
- reference `loss_curve`, `validation_scores`, accuracy, macro F1, and
  `confusion_matrix`
- same-seed replay and one different-seed sensitivity run
- undertraining and shuffled-label diagnoses plus smallest repairs
- the tiny-capacity comparison without a universal-capacity claim
- the code-reading trace of fit vs hold-out boundaries

See `missions/M21/evidence_contract.yaml`. Do not paste filled evidence into
the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M21/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams. The
repository intentionally leaves every learner response unfilled.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR and formal review

Use `missions/M21/adr_prompt.md` to author a black-box training acceptance
policy: acceptance metrics, seed/replay rules, monitoring, rollback, and
what blocks progression to M22.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR.


## M20 → M21 → M22 handoff

M20 monitors optimization. M21 trains and diagnoses the whole neural system
from split, budget, seed, labels, traces, and errors. M22 may open the
neuron and layer **only after** one accepted M21 run is defended.

M22 must not retroactively alter M21 held-out evidence to make internals
look successful. If you cannot explain the reference run without opening
weights, you are not ready to leave this mission.


## Mission summary prompt

In your own words, using only numbers and plots from this lab:

1. Why is the majority baseline the first number that matters?
2. What did the `loss_curve` and `validation_scores` show that a single
   accuracy number hides?
3. How did same-seed replay differ from the 2102 sensitivity run?
4. Why is `max_iter=1` a budget failure rather than a reason to inspect a
   layer?
5. Why is a shuffled-label run near baseline a data-system failure?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert holdout["majority_baseline_accuracy"] == reference.majority_baseline_accuracy
assert holdout["train_size"] == reference.train_size
assert reference.test_accuracy > 0.90
assert reference.macro_f1 > 0.90
assert reference.test_accuracy > reference.majority_baseline_accuracy
assert len(reference.loss_curve) >= 1
assert len(reference.validation_scores) >= 1
assert sum(map(sum, reference.confusion_matrix)) == reference.test_size
print("M21 integrity checks passed")
